In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
BG_COLOR = '#ffffff'
HIGHLIGHT_PLAYER = "Michael Olise"
HIGHLIGHT_COLOR = '#e31a1c'
CLUSTER_COLORS = ['#3498db', '#e74c3c', '#2ecc71']

FEATURES = ["Big_Chances_Created", "Assists", "Acc_Final_Third_Passes"]

# Set global plotting style
plt.rcParams['font.sans-serif'] = 'Arial'


In [ ]:
# ==========================================
# DATA LOADING
# ==========================================
def load_data(filepath="player_statistics.csv"):
    """Loads and returns the player statistics dataset from a CSV file."""
    try:
        # Read the CSV file into a Pandas DataFrame
        df = pd.read_csv(filepath)
        return df
    except FileNotFoundError:
        print(f"Error: The file '{filepath}' was not found.")
        return None
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return None


In [ ]:
# ==========================================
# MACHINE LEARNING
# ==========================================
def perform_clustering(df, n_clusters=3):
    """Applies K-Means clustering and adds cluster assignments to the DataFrame."""
    X = df[FEATURES]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Ask the machine to find distinct tactical profiles
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df['Cluster'] = kmeans.fit_predict(X_scaled)
    
    return df


In [ ]:
# ==========================================
# VISUALIZATIONS
# ==========================================
def plot_creator_matrix(df, output_path='creator_matrix.png'):
    """Generates a scatter plot of Progressions vs Lethal Creation."""
    fig, ax = plt.subplots(figsize=(11, 7.5), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    for idx, row in df.iterrows():
        cluster_id = row['Cluster']
        player_name = row['Player']
        
        # Highlight specific player
        if player_name == HIGHLIGHT_PLAYER:
            ax.scatter(row['Acc_Final_Third_Passes'], row['Big_Chances_Created'], 
                       color=HIGHLIGHT_COLOR, s=350, edgecolor='black', linewidth=2, zorder=5)
            ax.text(row['Acc_Final_Third_Passes'], row['Big_Chances_Created'] + 0.04, 
                    player_name, fontsize=11, fontweight='bold', ha='center', zorder=6)
        else:
            ax.scatter(row['Acc_Final_Third_Passes'], row['Big_Chances_Created'], 
                       color=CLUSTER_COLORS[cluster_id], s=120, alpha=0.8, edgecolor='white', linewidth=1, zorder=3)
            ax.text(row['Acc_Final_Third_Passes'], row['Big_Chances_Created'] + 0.03, 
                    player_name, fontsize=9, ha='center', alpha=0.8, zorder=4)

    # Medians for quadrant separation
    ax.axhline(y=df['Big_Chances_Created'].median(), color='#bdc3c7', linestyle='--', zorder=1)
    ax.axvline(x=df['Acc_Final_Third_Passes'].median(), color='#bdc3c7', linestyle='--', zorder=1)

    # Styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel("Progression (Accurate Final Third Passes per 90)", fontsize=12, fontweight='bold', labelpad=15)
    ax.set_ylabel("Lethal Creation (Big Chances Created per 90)", fontsize=12, fontweight='bold', labelpad=15)
    ax.set_title("The Creator Matrix", size=20, fontweight='bold', pad=25)

    plt.tight_layout()
    # fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_ml_clusters(df, output_path='ml_clusters.png'):
    """Plots a grouped bar chart showing the average metrics for each cluster."""
    fig, ax = plt.subplots(figsize=(11, 6), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    n_clusters = df['Cluster'].nunique()
    bar_width = 0.2
    index = np.arange(len(FEATURES))

    # Plot bars for each cluster's average stats
    for i in range(n_clusters):
        cluster_data = df[df['Cluster'] == i][FEATURES].mean()
        # Normalize the data for visual comparison (so Passes don't dwarf Big Chances)
        normalized_data = cluster_data / df[FEATURES].max() 
        
        ax.bar(index + i * bar_width, normalized_data, bar_width, 
               color=CLUSTER_COLORS[i], label=f'Cluster {i} Average', edgecolor='black')

    # Styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks(index + bar_width)
    ax.set_xticklabels(["Big Chances", "Assists", "Final 3rd Passes"], fontsize=11, fontweight='bold')
    ax.set_ylabel("Metric Strength (Normalized against Max)", fontsize=12, fontweight='bold', labelpad=15)
    ax.set_title("What did the Algorithm actually find?", size=20, fontweight='bold', pad=25)
    
    # Single legend execution
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=n_clusters)

    plt.tight_layout()
    # fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_top_assists(df, top_n=10, output_path='assists_comparison.png'):
    """Generates a horizontal bar chart comparing top players by assists."""
    # Extract the top N players and sort ascending for the horizontal bar chart
    df_sorted = df.nlargest(top_n, "Assists").sort_values(by="Assists", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG_COLOR) 
    ax.set_facecolor(BG_COLOR)

    # Dynamic color list
    bar_colors = [HIGHLIGHT_COLOR if player == HIGHLIGHT_PLAYER else '#b0bec5' for player in df_sorted['Player']]

    bars = ax.barh(df_sorted['Player'], df_sorted['Assists'], color=bar_colors, edgecolor='none', height=0.65)

    # Style adjustments
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#bdc3c7')
    ax.spines['bottom'].set_color('#bdc3c7')

    # Bold the target player on the y-axis ticks
    for tick in ax.get_yticklabels():
        if tick.get_text() == HIGHLIGHT_PLAYER:
            tick.set_fontweight('bold')

    # Data labels on bars
    for i, bar in enumerate(bars):
        width = bar.get_width()
        player_name = df_sorted.iloc[i]['Player']
        
        ax.text(
            width + 0.01,                     
            bar.get_y() + bar.get_height() / 2, 
            f'{width:.2f}',               
            va='center', ha='left', fontsize=9,
            fontweight='bold' if player_name == HIGHLIGHT_PLAYER else 'normal'
        )

    ax.set_xlabel("Assists per 90 mins", fontsize=12, fontweight='bold', labelpad=15)
    ax.set_title(f"Where {HIGHLIGHT_PLAYER.split()[-1]} Ranks in Assists", size=20, fontweight='bold', pad=15)

    plt.tight_layout()
    # fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

def print_cluster_rosters(df):
    """Prints out the players belonging to each cluster."""
    print("=== THE MACHINE LEARNING ROSTERS ===")
    n_clusters = df['Cluster'].nunique()
    
    for i in range(n_clusters):
        cluster_players = df[df['Cluster'] == i]['Player'].tolist()
        print(f"\nCluster {i} Players:")
        print(", ".join(cluster_players))


In [ ]:
# ==========================================
# MAIN EXECUTION PIPELINE
# ==========================================
def main():
    # 1. Load Data
    df = load_data()
    
    # 2. Process Machine Learning
    df = perform_clustering(df, n_clusters=3)
    
    # 3. Generate Visuals
    plot_creator_matrix(df)
    plot_ml_clusters(df)
    plot_top_assists(df, top_n=10)
    
    # 4. Print Summary
    print_cluster_rosters(df)

if __name__ == "__main__":
    main()
    